# Step 3. Create a custom retriever

In [1]:
# RAG-ретривер на базе TF-IDF (Term Frequency-Inverse Document Frequency) можно назвать частотным ретривером, 
# так как его работа напрямую основана на статистике встречаемости слов.

In [2]:
from __future__ import annotations
import os
import pickle
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple, Optional

from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

from gensim import corpora
from gensim.parsing import strip_tags, strip_numeric, \
    strip_multiple_whitespaces, stem_text, strip_punctuation, \
    remove_stopwords, preprocess_string, strip_non_alphanum
from gensim import models
from gensim import similarities
import pymorphy3

from retriever.stop_words import STOP_WORDS

In [3]:
morph_analyzer = pymorphy3.MorphAnalyzer()

In [4]:
SPLITS_PATH = './splits'
GENSIM_PATH = './gensim'

### Load TFIDF model files and splits

In [5]:
try:
    with open(os.path.join(GENSIM_PATH, 'dictionary.pkl'), 'rb') as h:
        dictionary = pickle.load(h)
    with open(os.path.join(GENSIM_PATH, 'similarity_index.pkl'), 'rb') as h:
        similarity_index = pickle.load(h)
    model_ = models.TfidfModel()
    retriever_model = model_.load(os.path.join(GENSIM_PATH, 'tfidf_model.mo'))
except FileNotFoundError:
    raise FileNotFoundError("Could not load gensim model files, please check gensim folder")

In [6]:
with open(os.path.join(SPLITS_PATH, 'splits.pkl'), 'rb') as f:
    splits = pickle.load(f)

### Query clening pipe

In [7]:
# Filters to be executed in pipeline
transform_to_lower = lambda s: s.lower()
CLEAN_FILTERS = [
                strip_tags,
                # strip_numeric,
                strip_punctuation,
                strip_non_alphanum,
                strip_multiple_whitespaces,
                transform_to_lower,
                ]

In [8]:
# Method does the filtering of all the unrelevant text elements
def cleaning_pipe(text:str) -> list[str]:
    # Invoking gensim.parsing.preprocess_string method with set of filters
    processed_words = preprocess_string(text, CLEAN_FILTERS)
    processed_words = [s for s in processed_words if len(s) > 1]
    processed_words = [s for s in processed_words if s not in STOP_WORDS]
    processed_words = [morph_analyzer.parse(s)[0].normal_form for s in processed_words]
    return processed_words

### Query engine for the custom retriever

In [9]:
def get_top_n(docs: List[Document], query: str, n: int, with_similarity: bool) -> List[Tuple[Document, float]] | List[Document] | None:
    """Retriever query engine
    Args:
        query: text query.
        n: number of returned documents.
    Returns:
        A list of tuples with relevance score.
    """
    query_bow = dictionary.doc2bow(query)
    sims = similarity_index[retriever_model[query_bow]]
    qty = sum(sims > 0)

    if qty > 0:
        top_idx = sims.argsort()[-1 * n:][::-1]
        result = []
        for idx in top_idx:
            similarity = round(float(sims[idx]), 3)
            doc = docs[idx]
            if with_similarity:
                result.append((doc, similarity))
            else:
                result.append(doc)
        return result
    else:
        return None

### Retriever template

In [10]:
class RetrieverTemplate(BaseRetriever):
    """Retriever template"""

    docs: List[Document]
    """Documents."""
    k: int = 3
    """Number of documents to return."""

    @classmethod
    def from_documents(
        cls,
        docs: Iterable[Document],
        **kwargs: Any,
    ) -> FreqRetriever:
        """
        Create a FreqRetriever instance from a list of langchain Documents.
        Args:
            docs: A list of of langchain Documents.
            **kwargs: Any other arguments to pass to the retriever.
        Returns:
            A Retriever instance.
        """
        return cls(
            docs=docs,
            **kwargs
        )

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Tuple[Document, float]] | None:
        
        return_docs = self.docs[:self.k]
        return return_docs

In [11]:
retriever = RetrieverTemplate.from_documents(
    docs=splits,
    k = 3,
)

result = retriever.invoke('')
for doc in result:
    print(doc.metadata)

{'title': 'ЖУК В МУРАВЕЙНИКЕ', 'id': '05fa57cd9e07c85d87b417cbe1fcd1c8', 'size': 93, 'collection': 'beetle_in_anthill'}
{'title': 'ЖУК В МУРАВЕЙНИКЕ', 'chapter': '1 июня 78–го года', 'section': 'СОТРУДНИК КОМКОНА–2 МАКСИМ КАММЕРЕР', 'id': '037c58a8f1011d66e00e2488bd45380a', 'size': 3853, 'collection': 'beetle_in_anthill'}
{'title': 'ЖУК В МУРАВЕЙНИКЕ', 'chapter': '1 июня 78–го года', 'section': 'Я принял папку. С такой я...', 'id': '9a490f7a02f67db0fa4bbb9710b1e9bb', 'size': 1676, 'collection': 'beetle_in_anthill'}


### Custom retriever class

In [12]:
class FreqRetriever(BaseRetriever):
    """TF-IDF retriever based on gensim model"""

    docs: List[Document]
    """Documents."""
    k: int = 4
    """Number of documents to return."""
    with_similarity: bool = False
    """True for return found chunk relevance"""

    @classmethod
    def from_documents(
        cls,
        docs: Iterable[Document],
        **kwargs: Any,
    ) -> FreqRetriever:
        """
        Create a FreqRetriever instance from a list of langchain Documents.
        Args:
            docs: A list of of langchain Documents.
            **kwargs: Any other arguments to pass to the retriever.
        Returns:
            A FreqRetriever instance.
        """
        return cls(
            docs=docs,
            **kwargs
        )

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Tuple[Document, float]] | None:
        processed_query = cleaning_pipe(query)
        return_docs = get_top_n(self.docs, processed_query, n=self.k, with_similarity=self.with_similarity)
        return return_docs

### Custom retriever test

In [13]:
retriever = FreqRetriever.from_documents(
    docs=splits,
    k = 3,
    with_similarity = True
)

In [14]:
def retriever_test(query: str) -> None:
    result = retriever.invoke(query)
    if not result:
        print("No relevant documents found")
        return None
    if hasattr(retriever, 'with_similarity') and retriever.with_similarity:
        for doc, similarity in result:
            print(f'Similarity: {similarity}')
            print(doc.metadata)
            print(f'{doc.page_content[:500]}...\n')
    else:
        for doc in result:
            print(doc.metadata)
            print(f'{doc.page_content[:500]}...\n')            

In [15]:
query1 = """Что напомнило Бромбергу о саркофаге в связи с Александром Дымком?"""
query2 = """На планете Саракш Абалкин участвовал в операции «Человек и голованы» в период с марта по июль какого года?"""

In [16]:
retriever_test(query1)

Similarity: 0.343
{'title': 'ЖУК В МУРАВЕЙНИКЕ', 'chapter': '4 июня 78–го года', 'section': 'Сегодня а вернее вчера...', 'id': 'b0e3d6cf3c5b77ef92b3daffcfba9a67', 'size': 4009, 'collection': 'beetle_in_anthill'}
4 июня 78–го года

Сегодня, а вернее, вчера утром к нему явился молодой человек лет сорока — сорока пяти и представился как Александр Дымок, конфигуратор сельскохозяйственной автоматики. Роста среднего, очень бледное лицо, длинные черные прямые волосы, как у индейца. Он пожаловался, что на протяжении многих месяцев пытается и никак не может выяснить обстоятельства исчезновения своих родителей. Он изложил Бромбергу в высшей степени загадочную и чертовски соблазнительную в своей загадочности леген...

Similarity: 0.166
{'title': 'ЖУК В МУРАВЕЙНИКЕ', 'chapter': '3 июня 78–го года', 'section': 'ЗАСТАВА НА РЕКЕ ТЕЛОН', 'id': '0087f722e0d74a421f48d1b567c2955b', 'size': 5302, 'collection': 'beetle_in_anthill'}
3 июня 78–го года

ЗАСТАВА НА РЕКЕ ТЕЛОН

Невидимая река шумела сквозь шурш

In [17]:
retriever_test(query2)

Similarity: 0.177
{'title': 'ЖУК В МУРАВЕЙНИКЕ', 'chapter': '01.06. — 13.01. СЛОН — СТРАННИКУ.', 'section': 'По–моему в этом сама...', 'id': 'f758c02debf2f9a5998f5e5c00a7cd33', 'size': 4065, 'collection': 'beetle_in_anthill'}
01.06. — 13.01. СЛОН — СТРАННИКУ.

По–моему, в этом сама суть Прогрессора: умение решительно разделить на своих и чужих. Именно за это умение дома к ним относятся с опасливым восторгом, с восторженной опаской, а сплошь и рядом — с несколько брезгливой настороженностью. И тут ничего не поделаешь. Приходится терпеть — и нам, и им. Потому что либо Прогрессоры, либо нечего Земле соваться во внеземные дела… Впрочем, к счастью, нам в КОМКОНе–2 достаточно редко приходится иметь дело с Прогрессорами.
Я ...

Similarity: 0.064
{'title': 'ЖУК В МУРАВЕЙНИКЕ', 'chapter': '01.06. — 13.01. СЛОН — СТРАННИКУ.', 'section': 'Ноябрь 66–го сентябрь...', 'id': 'd564622644bfdeca7230a2430ec264cc', 'size': 4703, 'collection': 'beetle_in_anthill'}
01.06. — 13.01. СЛОН — СТРАННИКУ.

Ноябрь 